<a href="https://colab.research.google.com/github/sanjivshukla1212-debug/PS_Project---Li-Ion-ML--sem-3/blob/main/ProjectPipeline1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q mp-api pymatgen pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 825.2 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.8/825.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━

In [10]:
import pandas as pd
import numpy as np
from google.colab import userdata

In [11]:
from mp_api.client import MPRester

In [12]:
API_KEY = userdata.get("MP_API_KEY")

In [13]:
with MPRester(API_KEY) as mpr:
  print("1. Connecting to the Materials Project API...")

  docs = mpr.materials.summary.search(
      elements = ["Li"],
      band_gap =(1.0, None),
      energy_above_hull = (0.0,0.05),
      fields=["material_id",
              "formula_pretty",
              "band_gap",
              "energy_above_hull",
              "structure"
              ]
  )
print(f"-> Extraction complete! Retrieved {len(docs)} initial structures. ")


1. Connecting to the Materials Project API...


Retrieving SummaryDoc documents:   0%|          | 0/5133 [00:00<?, ?it/s]

-> Extraction complete! Retrieved 5133 initial structures. 


In [14]:
print("\n2. Building the Pandas Dataframe...")
extracted_data  =  []
for doc in docs:
  extracted_data.append({
      "MPID": str(doc["material_id"]),
      "Formula": doc.formula_pretty,
      "BandGap_eV": round(doc.band_gap,3) if doc.band_gap else None,
      "E_hull": round(doc.energy_above_hull,4) if doc.energy_above_hull else None,
      "Structure": doc.structure
  })
  df = pd.DataFrame(extracted_data)


2. Building the Pandas Dataframe...


In [15]:
print("\n3. Pruning structure containing transition metals...")

def contains_transistion_metal(structures):
  for el in structures.composition.elements:
    if el.is_transition_metal:
      return True
  return False

mask = ~df["Structure"].apply(contains_transistion_metal)
df_filtered = df[mask].copy().reset_index(drop=True)

print(f"  => Materials before filtering: {len(df)}")
print(f"  => Materials after filtering: {len(df_filtered)}\n")


3. Pruning structure containing transition metals...
  => Materials before filtering: 5133
  => Materials after filtering: 1201



In [16]:
def extract_sendek_features(structures):
  li_indices = [i for i, site in enumerate(structures) if site.specie.symbol == "Li"]
  anion_symbols = {"O","S","Se","Te","F","Cl","Br","I","N","P"}
  anion_indices = [i for i,site in enumerate(structures) if site.specie.symbol in anion_symbols]

  if not li_indices or not anion_indices:
    return None

  li_anion_dists=[]
  li_li_dists=[]
  li_neighbor_counts = []

  for i in li_indices:
    all_nbrs = structures.get_neighbours(structures[i], r = 8.0)
    anion_nbrs = [nbr for nbr in all_nbrs if nbr.specie.symbol in anion_symbols]
    if anion_nbrs:
        li_anion_dists.append(min(nbr.nn_distance for nbr in anion_nbrs))


        other_li_nbrs = [nbr for nbr in all_nbrs if nbr.specie.symbol == "Li"]
        if other_li_nbrs:
            li_li_dists.append(min(nbr.nn_distance for nbr in other_li_nbrs))

            li_neighbor_counts.append(sum(1 for nbr in other_li_nbrs if nbr.nn_distance <= 4.0))

    if not li_anion_dists or not li_li_dists:
        return None
    lasd = float(np.mean(li_anion_dists))
    llsd = float(np.mean(li_li_dists))
    llb = float(np.mean(li_neighbor_counts)) if li_neighbor_counts else 0.0
    non_li_sites = [s for s in structures if s.specie.symbol != "Li"]
    ionicities = []
    for s in non_li_sites:

        nbrs = structures.get_neighbors(s, r=3.2)
        for nbr in nbrs:
            if nbr.specie.symbol != "Li":
                chi_a = getattr(s.specie, 'X', None)
                chi_b = getattr(nbr.specie, 'X', None)
                if chi_a is not None and chi_b is not None:

                    ionicity = 1.0 - np.exp(-0.25 * ((chi_a - chi_b) ** 2))
                    ionicities.append(ionicity)
    sbi = float(np.mean(ionicities)) if ionicities else 0.5

    # 4. Anion Framework Coordination (AFC)
    anion_coords = []
    for a_idx in anion_indices:
        framework_nbrs = [nbr for nbr in structures.get_neighbors(structures[a_idx], r=3.2) if nbr.specie.symbol != "Li"]
        anion_coords.append(len(framework_nbrs))
    afc = float(np.mean(anion_coords)) if anion_coords else 4.0

    return {
        "LASD": round(lasd, 3),
        "LLSD": round(llsd, 3),
        "LLB": round(llb, 2),
        "SBI": round(sbi, 3),
        "AFC": round(afc, 2)
    }

In [17]:
import pandas as pd
import numpy as np # numpy is required for np.mean

def extract_sendek_features(structures):
  li_indices = [i for i, site in enumerate(structures) if site.specie.symbol == "Li"]
  anion_symbols = {"O","S","Se","Te","F","Cl","Br","I","N","P"}
  anion_indices = [i for i,site in enumerate(structures) if site.specie.symbol in anion_symbols]

  if not li_indices or not anion_indices:
    return None

  li_anion_dists=[]
  li_li_dists=[]
  li_neighbor_counts = []

  for i in li_indices:
    all_nbrs = structures.get_neighbors(structures[i], r = 8.0) # Fixed: changed get_neighbours to get_neighbors
    anion_nbrs = [nbr for nbr in all_nbrs if nbr.specie.symbol in anion_symbols]
    if anion_nbrs:
        li_anion_dists.append(min(nbr.nn_distance for nbr in anion_nbrs))

        # Find the closest adjacent Lithium (LLSD) and count nearby available hopping sites (LLB)
        other_li_nbrs = [nbr for nbr in all_nbrs if nbr.specie.symbol == "Li"]
        if other_li_nbrs:
            li_li_dists.append(min(nbr.nn_distance for nbr in other_li_nbrs))
            # LLB uses a standard 4.0 Å bounding sphere for local coordination
            li_neighbor_counts.append(sum(1 for nbr in other_li_nbrs if nbr.nn_distance <= 4.0))

  # Fixed: Moved this check outside the for loop to ensure all Li sites are processed before deciding to return None
  if not li_anion_dists or not li_li_dists:
      return None

  lasd = float(np.mean(li_anion_dists))
  llsd = float(np.mean(li_li_dists))
  llb = float(np.mean(li_neighbor_counts)) if li_neighbor_counts else 0.0
  non_li_sites = [s for s in structures if s.specie.symbol != "Li"]
  ionicities = []
  for s in non_li_sites:
      # A tighter 3.2 Å radius captures the immediate rigid framework cage
      nbrs = structures.get_neighbors(s, r=3.2)
      for nbr in nbrs:
          if nbr.specie.symbol != "Li":
              chi_a = getattr(s.specie, 'X', None)
              chi_b = getattr(nbr.specie, 'X', None)
              if chi_a is not None and chi_b is not None:
                  # Pauling electronegativity calculation
                  ionicity = 1.0 - np.exp(-0.25 * ((chi_a - chi_b) ** 2))
                  ionicities.append(ionicity)
  sbi = float(np.mean(ionicities)) if ionicities else 0.5

  # 4. Anion Framework Coordination (AFC)
  anion_coords = []
  for a_idx in anion_indices:
      framework_nbrs = [nbr for nbr in structures.get_neighbors(structures[a_idx], r=3.2) if nbr.specie.symbol != "Li"]
      anion_coords.append(len(framework_nbrs))
  afc = float(np.mean(anion_coords)) if anion_coords else 4.0

  return {
      "LASD": round(lasd, 3),
      "LLSD": round(llsd, 3),
      "LLB": round(llb, 2),
      "SBI": round(sbi, 3),
      "AFC": round(afc, 2)
  }


print(f"Extracting features for {len(df_filtered)} materials...")

# 1. Broadcast your function across the entire 'Structure' column
# This creates a single new column filled with dictionaries (or None)
df_filtered['extracted_dict'] = df_filtered['Structure'].apply(extract_sendek_features)

# 2. Purge the failures
# If a crystal lacked Lithium or valid anions, your function returned None. Drop those rows.
df_clean = df_filtered.dropna(subset=['extracted_dict']).copy()
print(f"Extraction successful for {len(df_clean)} materials.")

# 3. The Dictionary Expansion Trick
# apply(pd.Series) automatically unpacks the dictionary keys into brand new DataFrame columns
feature_cols = df_clean['extracted_dict'].apply(pd.Series)

# 4. Merge and Clean Up
# Concatenate the original dataframe with your 5 new feature columns,
# and drop the temporary dictionary column.
df_final = pd.concat([df_clean.drop(columns=['extracted_dict']), feature_cols], axis=1)

# Display the final, ML-ready DataFrame
display(df_final.head())

Extracting features for 1201 materials...
Extraction successful for 1127 materials.


,MPID,Formula,BandGap_eV,E_hull,Structure,LASD,LLSD,LLB,SBI,AFC
0,mp-3346453,LiMgCl3,6.594,0.0143,"[[3.148772 1.81794431 3.26062522] Li, [-1.46...",2.475,3.705,3.0,0.575,2.0
1,mp-3346409,Na4LiCl5,6.184,0.0129,[[5.19356794e+00 1.74609233e+00 3.99041042e-16...,2.635,5.521,0.0,0.712,4.8
2,mp-3346553,Na2LiCl3,6.280,0.0246,[[-6.68478182e-17 -6.68478182e-17 5.42085890e...,2.603,3.877,4.0,0.712,4.0
3,mp-3347048,Na4LiCl5,6.209,0.0120,"[[10.07041613 1.93891749 0.02758599] Na, [ 8...",2.654,3.878,2.0,0.712,4.8
4,mp-3346730,Na2LiCl3,6.306,0.0269,[[1.62965569e+01 0.00000000e+00 8.98120354e-16...,2.655,4.644,2.0,0.712,4.0
